In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
import ase
import os
import ase.io
from tqdm.notebook import tqdm
tqdm.pandas()

In [2]:
def get_a2f(file_name,num_atoms):
    cm_to_eV = 13.605693122994
    
    with open(file_name,"r") as fp:
        lines = fp.readlines()
        
    freq = []
    tot_phdos = []
    site_proj_phdos = [[] for i in range(num_atoms*3)]
    for i in range(5,len(lines)-1):
        line = lines[i].split()
        freq.append(float(line[0])*cm_to_eV*1000)
        tot_phdos.append(float(line[1]))#/(cm_to_eV*1000))
        for j in range(0,num_atoms*3):
            site_proj_phdos[j].append(float(line[2+j])/(cm_to_eV*1000))
        
    return freq, tot_phdos, site_proj_phdos

def cal_lamb(freq_w,alpha_F):
    lambdaF = 0
    # try:
    for i in range(1,len(freq_w)):
        if freq_w[i]>0:
            dw = freq_w[i] - freq_w[i-1]
            w = freq_w[i]
            alpha_F_w = alpha_F[i]
            lambdaF = lambdaF + ((alpha_F_w/w)*dw)
    return 2*lambdaF

def cal_w_log(freq_w,alpha_F,lamb):
    w_logF = 0
    try:
        i = 1
        for i in range(1,len(freq_w)):
            if freq_w[i]>0:            
                dw = freq_w[i] - freq_w[i-1]
                w_logF = w_logF + (alpha_F[i]*np.log(freq_w[i])*dw/freq_w[i])
        return np.exp(2*w_logF/lamb)/0.08617
    except: 
        return np.nan          

def cal_w_sq(freq_w,alpha_F,lamb):
    w_sqF = 0
    try:
        for i in range(1,len(freq_w)):
            if freq_w[i]>0:
                dw = freq_w[i] - freq_w[i-1]
                w_sqF = w_sqF + (alpha_F[i]*freq_w[i]*dw)
        return ((2*w_sqF/lamb)**.5)/0.08617
    except:
        return np.nan

def cal_tc(lamb, omega_log,mu=0.1):
    frac = -1.04*(1+lamb)/(lamb-mu*(1+0.62*lamb))
    return (omega_log/1.2)*np.exp(frac)    

def cal_tc_ad(lamb,wlog,w2,tc,mu=0.1):
    f1 = (1+(lamb/(2.46*(1+3.8*mu)))**1.5)**(1/3)
    f2 = 1+((lamb**2)*((w2/wlog)-1))/(lamb**2+(1.82*(1+6.3*mu)*(w2/wlog))**2)
    return f1*f2*tc

In [3]:
from scipy.signal import savgol_filter


In [4]:
def get_tot_a2F(df):
    jobdir = f'{root}{df.name}/'
    # if os.path.isdir(jobdir+'run_2'):
    #     jobdir = jobdir + 'run_2/'
    filename = jobdir+"/relax.out"
    atom_obj = ase.io.read(filename)
    if os.path.isfile(jobdir+'/a2F.dos20'):
        filename = jobdir+"/relax.out"
        atom_obj = ase.io.read(filename)
        freq, phdos, site_proj_phdos = get_freq_phonon_2x2x2_interpolated_DOS(jobdir +"/a2F.dos20", atom_obj.get_global_number_of_atoms())
    else:
        phdos = np.nan
    return phdos 
    
def get_freq(df):
    jobdir = f'{root}{df.name}/'
    if os.path.isdir(jobdir+'run_2'):
        jobdir = jobdir + 'run_2/'
    filename = jobdir+"/relax.out"
    atom_obj = ase.io.read(filename)
    if os.path.isfile(jobdir+'/a2F.dos20'):
        filename = jobdir+"/relax.out"
        atom_obj = ase.io.read(filename)
        freq, phdos, site_proj_phdos = get_freq_phonon_2x2x2_interpolated_DOS(jobdir +"/a2F.dos20", atom_obj.get_global_number_of_atoms())
    else:
        freq = np.nan
    return freq

def get_target(df):    
    x = df.freq_a2f
    y = df.a2f
    xl = np.arange(0.25,101,0.1)
    y =  np.interp(xl,x,y) 
    Y = savgol_filter(y,101,3, mode='interp')
    #xll = np.arange(0.25,101,2.)
    Y = np.interp(Freq_final,xl,Y)     
    Y = np.asarray([y if y >0.0 else 0. for y in Y])
    return Y#/max(Y)

def get_freq_smooth(df):
    return Freq_final
Freq_final =np.arange(0.25,101,2)


In [5]:
par_el = 'run_full'
root = '/blue/hennig/jasongibson/diff_model'
root = f'{root}/materials/{par_el}/mp_relaxed/'
df = pd.read_pickle(f'pkl_files/df_cpd_{par_el}_strict.pkl')
df = df.loc[df.tcad_cpd >= 5]

In [6]:
def get_a2f_properties(df):
    jobdir = f'{root}{df.name}/'
    # if df.name == 1601:
    #     jobdir = jobdir + 'high/'  
    #     print('here')
    # elif os.path.isdir(jobdir+'run_2'):
    #     jobdir = jobdir + 'run_2/'    
    filename = jobdir + "/relax.out"
    
    try:
        atom_obj = ase.io.read(filename)
    except:
        return pd.Series([np.nan, np.nan], index=['freq_a2f', 'a2f'])
    
    if os.path.isfile(jobdir + 'a2F.dos20'):
        filename = jobdir + "/relax.out"
        atom_obj = ase.io.read(filename)
        freq, a2f, site_proj_a2f = get_a2f(jobdir +"a2F.dos20", atom_obj.get_global_number_of_atoms())
    else:
        freq, a2f, site_proj_a2f = np.nan, np.nan, np.nan
    
    return pd.Series([freq, a2f], index=['freq_a2f', 'a2f'])

df[['freq_a2f', 'a2f']] = df.apply(get_a2f_properties, axis=1)
# df[['freq_a2f_2', 'a2f_2']] = df.progress_apply(get_a2f_properties, axis=1)


/scratch/local/65299106/ipykernel_564693/990544361.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[['freq_a2f', 'a2f']] = df.apply(get_a2f_properties, axis=1)
/scratch/local/65299106/ipykernel_564693/990544361.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[['freq_a2f', 'a2f']] = df.apply(get_a2f_properties, axis=1)


In [7]:
print(df.shape)
df.dropna(axis=0,inplace=True)
print(df.shape)
# df['a2f'] = df.apply(get_target,axis=1)
# df['freq_a2f'] = df.apply(get_freq_smooth,axis=1)

def get_size(df):
    return len(df.structure)
df['natoms'] = df.apply(get_size, axis = 1)

(1363, 139)
(682, 139)


In [8]:
mu = 0.1
def get_lamb(df):
    try:
        return cal_lamb(df.freq_a2f,df.a2f)
    except:
        return np.nan

def get_w2(df):
    try:
        return cal_w_sq(df.freq_a2f,df.a2f,df.lamb_final)
    except:
        return np.nan

def get_wlog(df):
    try:
        return cal_w_log(df.freq_a2f,df.a2f,df.lamb_final)
    except:
        return np.nan

def get_tc(df):
    try:
        return cal_tc(df.lamb_final,df.wlog_final,mu)
    except:
        return np.nan

def get_tcad(df):
    try:
        return cal_tc_ad(df.lamb_final,df.wlog_final,df.w2_final,df.tc_final,mu)
    except:
        return np.nan

In [9]:
df['lamb_final'] = df.apply(get_lamb,axis=1)
df['w2_final'] = df.apply(get_w2,axis=1)
df['wlog_final'] = df.apply(get_wlog,axis=1)
df['tc_final'] = df.apply(get_tc,axis=1)
df['tcad_final'] = df.apply(get_tcad,axis=1)
df.sort_values('tcad_final', ascending = False,inplace=True)

/scratch/local/65299106/ipykernel_564693/1893784244.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['lamb_final'] = df.apply(get_lamb,axis=1)
/scratch/local/65299106/ipykernel_564693/1893784244.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['w2_final'] = df.apply(get_w2,axis=1)
/scratch/local/65299106/ipykernel_564693/1893784244.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at onc

In [10]:
def get_nelements(df):
    return len(set(df.structure.get_chemical_symbols()))
df['nelements'] = df.apply(get_nelements,axis=1)

/scratch/local/65299106/ipykernel_564693/900407074.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['nelements'] = df.apply(get_nelements,axis=1)


In [12]:
df = df.loc[df.tcad_final>=5]#.loc[df.mp_eah <0.05][['formula','sgn','tcad_final','mp_eah']]#.head(20)
len(df)

612

In [18]:
df[['formula','tcad_final','sgn','mp_eah']]

,formula,tcad_final,sgn,mp_eah
872,GaNb6Si,36.424072,200,0.092368
7807,Re2Ru2,31.754612,123,0.139794
101256,Nb5Si2V,29.341672,1,0.121486
56153,HNNb6Sn2,28.157370,225,0.195628
5378,GaNb6Zn,28.148405,200,0.107763
...,...,...,...,...
133470,CMo2Tc,5.149593,8,0.061254
49806,Hf2HgNb,5.147089,123,0.066593
85339,Nb4ReTi,5.120639,1,0.030215
146616,Ir2Nb2,5.079045,11,0.024972


In [18]:
from mp_api.client import MPRester

from pymatgen.core import Structure, Element, Composition
from pymatgen.analysis.phase_diagram import PhaseDiagram, PDEntry,PDPlotter
from pymatgen.ext.matproj import MPRester
from pymatgen.io.vasp import Vasprun
import os
from tqdm.notebook import tqdm
import pandas as pd

from pymatgen.io.vasp.outputs import Vasprun, Outcar, Potcar
from pymatgen.entries.computed_entries import ComputedEntry, ComputedStructureEntry
from pymatgen.entries.compatibility import MaterialsProject2020Compatibility

import re

from matplotlib import pyplot as plt

In [19]:
def sort_elements(formula):    # Extract elements using a regular expression
    elements = re.findall(r'[A-Z][a-z]*', formula)
    # Sort the elements alphabetically
    sorted_elements = ''.join(sorted(elements))
    return sorted_elements

In [20]:
api_org = '7JGUQgNZyOTTp8Tc'
api = '5nXBXhf4OGFQDbCo6BAyywL7yEp5RuQX'
compat = MaterialsProject2020Compatibility()
df['Sorted_Elements'] = df['formula'].apply(sort_elements)

# Sort the dataframe by the sorted elements
df.sort_values(by='Sorted_Elements',inplace=True)

In [21]:
root_relaxed = f'/blue/hennig/jasongibson/diff_model/materials/run_full/mp_relaxed'#/{par_el}/mp_relaxed/'

In [22]:
inds = []
eahs = []
cur_elements = {}

with MPRester(api) as mpr:
    for d, row in tqdm(df.iterrows(),total=len(df)):
        vr = Vasprun(f'{root_relaxed}/{d}/vasprun.xml') 
        struct = vr.final_structure
        species =list(set([spec.name for spec in struct.species]))            
        elements = struct.symbol_set
          
        if cur_elements != elements:
            entries = mpr.get_entries_in_chemsys(elements)
            cur_elements = elements
            phasediagram = PhaseDiagram(entries)    
        else:
            print('same?')
        
        potcar = Potcar.from_file(f'{root_relaxed}/{d}/POTCAR')
        potcar_symbols = [f'PBE {p.symbol}' for p in potcar]
        pde = ComputedStructureEntry(vr.final_structure,vr.final_energy)
        pde.parameters = {'run_type': 'GGA', 'potcar_symbols': potcar_symbols}  
        compat.process_entry(pde)
        
        eah = phasediagram.get_e_above_hull(pde,allow_negative=True)
        print(f'{row.formula}\t {eah:.4f}')
        inds.append(int(d))
        eahs.append(eah)

  0%|          | 0/612 [00:00<?, ?it/s]

/blue/hennig/jasongibson/envs/e3nn/lib/python3.9/site-packages/pymatgen/io/vasp/outputs.py:310: EncodingWarning: We strongly encourage explicit `encoding`, and we would use UTF-8 by default as per PEP 686
  with zopen(filename, mode="rt") as file:
/blue/hennig/jasongibson/envs/e3nn/lib/python3.9/site-packages/pymatgen/io/vasp/inputs.py:2664: EncodingWarning: We strongly encourage explicit `encoding`, and we would use UTF-8 by default as per PEP 686
  with zopen(filename, mode="rt") as file:


Retrieving ThermoDoc documents:   0%|          | 0/67 [00:00<?, ?it/s]

/blue/hennig/jasongibson/envs/e3nn/lib/python3.9/site-packages/pymatgen/entries/compatibility.py:1156: UserWarning: Failed to guess oxidation states for Entry None (HfNb2AlH). Assigning anion correction to only the most electronegative atom.
  warnings.warn(


HAlHfNb2	 0.1028


...

TcTiV2	 0.1710
same?
TcTiV2	 0.1628


Retrieving ThermoDoc documents:   0%|          | 0/29 [00:00<?, ?it/s]

TiV2W	 0.1455


Retrieving ThermoDoc documents:   0%|          | 0/13 [00:00<?, ?it/s]

V3W	 0.0967


In [23]:
df_mp = pd.DataFrame(eahs)
df_mp.index = inds
df_mp.columns = ['mp_eah']
df['mp_eah_cor']=df_mp.mp_eah

In [33]:
a = d.structure_dict.values[0]


In [34]:
a

{'numbers': array([41, 41, 41, 41, 41, 41, 31, 77]),
 'positions': array([[2.78686766, 0.48026596, 4.37493329],
        [4.08682716, 3.03741986, 3.11765864],
        [2.78688406, 0.48027487, 1.86033163],
        [0.22995534, 4.33722139, 0.56054841],
        [0.22994301, 1.7375565 , 0.56054002],
        [1.48718837, 3.03742626, 3.11765009],
        [2.78734748, 3.03758075, 0.56059279],
        [0.22965109, 0.48031164, 3.11769476]]),
 'cell': array([[ 5.11424138e+00, -1.41722832e-05,  2.35496279e-05],
        [ 2.86866917e-05,  5.11425861e+00,  2.24299449e-05],
        [-3.06616880e-05, -2.98893142e-05,  5.11428021e+00]]),
 'pbc': array([ True,  True,  True])}

In [30]:
def to_dict(df):
    return df.structure.todict()
df['structure_dict'] = df.apply(to_dict,axis = 1)

In [ ]:
d = df.loc[df.tcad_final>20].loc[df.mp_eah_cor<=.08].loc[~df.formula.str.contains('Tc')].loc[~df.formula.str.contains('Cr')].loc[~df.formula.str.contains('Be')]
# d = d.loc[~df.formula.str.contains('Mn')].loc[~df.formula.str.contains('Fe')].loc[~df.formula.str.contains('Co')].loc[~df.formula.str.contains('Ni')]
# d = d.loc[~df.formula.str.contains('Ni')].loc[~df.formula.str.contains('Cu')].loc[~df.formula.str.contains('Zn')].loc[~df.formula.str.contains('O')]
# d = d.loc[d.par_el!='mp_data'].loc[d.par_el!='alex']

In [40]:
df.sort_values('tcad_final',inplace=True,ascending=False)

In [44]:
df[['formula','tcad_final','mp_eah_cor','sgn','structure_dict']].to_csv('diff_model.csv')#.loc[df.mp_eah_cor<=.08]

In [28]:
df.loc[df.mp_eah_cor<0.05].loc[df.tcad_final>15][['formula','sgn','lamb_final','tcad_final','mp_eah','mp_eah_cor','a2f']]

,formula,sgn,lamb_final,tcad_final,mp_eah,mp_eah_cor,a2f
82587,AlNb6Rh,200,1.281002,19.078750,0.049440,0.049495,"[1.07518e-08, 2.92828e-07, 1.35804e-06, 3.7292..."
51254,AuNb6Pt,200,1.438564,15.863476,-0.000736,-0.000736,"[1.30576e-07, 3.52763e-06, 1.63335e-05, 4.4821..."
122437,Be4RhTa,216,0.943644,20.496278,0.035234,0.035249,"[1.94779e-09, 5.29475e-08, 2.4546e-07, 6.73934..."
11718,GaIrNb6,200,1.420186,24.785654,0.021650,0.021724,"[5.8044e-09, 1.57871e-07, 7.31958e-07, 2.00975..."
173836,GaMo6Re,200,1.002289,17.712669,0.008487,0.009526,"[5.96666e-09, 1.62e-07, 7.50838e-07, 2.06129e-..."
168623,GeIrMo4Nb2,6,1.203312,17.795314,0.048025,0.049932,"[1.31645e-08, 3.58503e-07, 1.66261e-06, 4.5655..."
72557,H2Mo2,4,0.914978,16.703584,0.006163,-0.083324,"[1.83722e-09, 4.96086e-08, 2.29673e-07, 6.3022..."
167606,H4Mo4,59,0.921236,16.783657,0.005719,-0.083768,"[1.54106e-09, 4.16872e-08, 1.93069e-07, 5.2986..."
580,H2Mo2,71,0.935820,17.241033,0.005679,-0.083808,"[1.38945e-09, 3.83173e-08, 1.78154e-07, 4.8975..."
53971,H2Mo2,11,0.928837,17.066853,0.005988,-0.083499,"[1.47609e-09, 3.98628e-08, 1.84558e-07, 5.0643..."


In [28]:
df.loc[df.mp_eah<0.05].loc[df.tcad_final>15][['formula','sgn','lamb_final','tcad_final','mp_eah','a2f']]

,formula,sgn,lamb_final,tcad_final,mp_eah,a2f
11718,GaIrNb6,200,1.420186,23.942819,0.021650,"[5.8044e-09, 1.57871e-07, 7.31958e-07, 2.00975..."
20657,Mo2NbTc3,164,1.417800,21.121194,-0.008514,"[4.71607e-09, 1.29729e-07, 6.02842e-07, 1.6568..."
122437,Be4RhTa,216,0.943644,19.411302,0.035234,"[1.94779e-09, 5.29475e-08, 2.4546e-07, 6.73934..."
13944,Mo2NbTc3,65,1.307707,19.196684,-0.011938,"[1.49713e-08, 4.06602e-07, 1.88464e-06, 5.1740..."
149313,Mo5NbOsRu,25,1.250162,18.546723,0.003758,"[1.02386e-08, 2.76976e-07, 1.28279e-06, 3.5205..."
146872,ReTc3Zr2,166,1.355299,18.454703,0.043219,"[2.31442e-08, 6.29152e-07, 2.91671e-06, 8.0081..."
538,Nb2Tc4,227,1.383863,18.397585,0.025132,"[1.78575e-08, 4.82686e-07, 2.23515e-06, 6.1338..."
82587,AlNb6Rh,200,1.281002,18.360329,0.049440,"[1.07518e-08, 2.92828e-07, 1.35804e-06, 3.7292..."
37068,Mo5OsRhW,1,1.471705,17.453285,0.018439,"[4.37408e-08, 1.20632e-06, 5.60887e-06, 1.5419..."
168623,GeIrMo4Nb2,6,1.203312,17.081529,0.048025,"[1.31645e-08, 3.58503e-07, 1.66261e-06, 4.5655..."


In [12]:
df[['formula','sgn','lamb_final','tcad_final','mp_eah','a2f']]

,formula,sgn,lamb_final,tcad_final,mp_eah,a2f
872,GaNb6Si,200,2.409011,36.424072,0.092368,"[7.81282e-08, 2.15002e-06, 9.99368e-06, 2.7470..."
7807,Re2Ru2,123,2.639579,31.754612,0.139794,"[2.83639e-09, 7.66565e-08, 3.5496e-07, 9.7409e..."
101256,Nb5Si2V,1,1.703267,29.341672,0.121486,"[9.60667e-09, 2.76836e-07, 1.29858e-06, 3.5834..."
56153,HNNb6Sn2,225,2.587249,28.157370,0.195628,"[1.17567e-07, 3.25363e-06, 1.51378e-05, 4.1626..."
5378,GaNb6Zn,200,6.720966,28.148405,0.107763,"[0.000143483, 0.00387421, 0.0179363, 0.0492174..."
...,...,...,...,...,...,...
78354,IrReRuW,47,0.447737,1.602397,0.059836,"[1.24174e-09, 3.35369e-08, 1.55273e-07, 4.2607..."
112615,Nb5Ru,123,0.434899,1.435584,0.017927,"[5.62511e-09, 1.57604e-07, 7.35157e-07, 2.0238..."
19670,Ir3Nb2W,10,0.443354,1.415766,0.022179,"[3.44841e-09, 9.51358e-08, 4.42411e-07, 1.2163..."
199225,HfIrTa2,25,0.430950,1.027123,0.112113,"[3.81501e-09, 1.03135e-07, 4.77597e-07, 1.3106..."


In [13]:
len(d)/len(df)

0.9078694817658349

In [15]:

df.dropna(axis=0,inplace=True)
df.to_pickle(f'pkl_files/df_a2f_{par_el}.pkl')

In [16]:
df

,dirs,Structure,final_E,PBE,GLLB-SC,HSE,SCAN,E_form,is_metal,formula,...,tc_cpd,tcad_cpd,freq_a2f,a2f,lamb_final,w2_final,wlog_final,tc_final,tcad_final,nelements
872,tc10_w2,"[[2.90378492 0.77667463 1.34814213] Nb, [0.365...",-71.214394,-0.009433,0.423296,-0.009309,0.081636,-0.234222,True,GaNb6Si,...,13.782114,14.760680,"[0.04513729510632628, 0.1451387313895385, 0.24...","[7.81282e-08, 2.15002e-06, 9.99368e-06, 2.7470...",2.409011,217.437125,181.816830,29.342537,36.424072,3
7807,tc10_w2,"[[0.34196889 2.16212195 2.87507346] Re, [2.252...",-43.804409,-0.008724,0.627173,-0.007106,0.133005,-0.066947,True,Re2Ru2,...,8.203227,8.506391,"[0.049924322174720495, 0.14992521423020777, 0....","[2.83639e-09, 7.66565e-08, 3.5496e-07, 9.7409e...",2.639579,162.815345,153.753357,26.046834,31.754612,2
101256,tc15_w2,"[[4.12043419 4.81051645 0.07742249] Nb, [1.591...",-72.877380,-0.008744,0.298951,-0.008966,0.106423,-0.319320,True,Nb5Si2V,...,10.375791,10.897899,"[0.04428517054603317, 0.14428565443072677, 0.2...","[9.60667e-09, 2.76836e-07, 1.29858e-06, 3.5834...",1.703267,231.156650,202.067375,25.766918,29.341672,3
56153,tc10_w2,"[[3.29710942 0.96038914 4.64531168] Nb, [3.250...",-84.459213,-0.009798,0.771711,-0.007947,0.218538,-0.521153,True,HNNb6Sn2,...,11.454735,12.364384,"[0.0443680292171522, 0.14436864915877703, 0.24...","[1.17567e-07, 3.25363e-06, 1.51378e-05, 4.1626...",2.587249,185.006843,126.302315,21.178923,28.157370,4
5378,tc10_w2,"[[1.71599546 0.41171237 1.47503818] Nb, [1.702...",-65.616768,-0.008630,0.427390,-0.009135,0.201362,-0.133445,True,GaNb6Zn,...,17.198797,19.163834,"[0.04799544306067363, 0.14799592694536726, 0.2...","[0.000143483, 0.00387421, 0.0179363, 0.0492174...",6.720966,98.711696,51.560445,11.777536,28.148405,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133470,tc15_w2,"[[1.31210972 1.33144627 4.77798832] Tc, [3.252...",-41.443390,-0.007350,0.484258,-0.007372,0.545039,-0.069232,True,CMo2Tc,...,9.942675,10.492609,"[0.049713161817451626, 0.14971432598680137, 0....","[3.27177e-09, 8.85729e-08, 4.10278e-07, 1.1260...",0.587454,312.075116,233.043580,4.995394,5.149593,3
49806,tc10_w2,"[[1.53723404 0.59014326 3.17280728] Hf, [1.538...",-30.636829,-0.007102,0.568606,-0.007532,0.059646,-0.120390,True,Hf2HgNb,...,5.077127,5.285684,"[0.04828007416080667, 0.1482802859316378, 0.24...","[2.05465e-08, 5.5997e-07, 2.59735e-06, 7.13289...",0.719315,161.981893,133.323412,4.949118,5.147089,3
85339,tc10_w2,"[[3.56034587 0.84134195 5.1662875 ] Ti, [1.314...",-61.209442,-0.010392,0.349277,-0.009530,0.033407,-0.031504,True,Nb4ReTi,...,5.334860,5.492028,"[0.04740413963754831, 0.14740543986382929, 0.2...","[3.90422e-09, 1.06414e-07, 4.93597e-07, 1.3555...",0.600064,245.877130,217.645211,4.979670,5.120639,3
146616,tc15_w2,"[[0.77557202 0.86734 2.84897693] Nb, [0.303...",-40.064507,0.073553,0.525314,0.052077,0.385574,-0.506016,True,Ir2Nb2,...,7.189223,7.494024,"[0.04990949196921643, 0.1499102479677725, 0.24...","[8.09089e-09, 2.18561e-07, 1.01196e-06, 2.7769...",0.644118,200.805972,175.451884,4.921661,5.079045,2


In [26]:
df.loc[[1608,1664,1601]][['formula','tcad_final','mp_eah']].head()

KeyError: "None of [Index([1608, 1664, 1601], dtype='int64')] are in the [index]"

In [ ]:
# df['tcad_final'] = df.tcad_final.values.real.astype(np.longdouble)

In [ ]:
# ind = 176565
# plt.plot(df.loc[ind].freq_a2f,np.asarray(df.loc[ind].a2f),'--',label='a2F')
# # plt.plot(df.loc[ind].freq_a2f_2,np.asarray(df.loc[ind].a2f_2),'--',label='a2F_2')

# plt.plot(df.loc[ind].freq,np.asarray(df.loc[ind].phdos)/max(df.loc[ind].phdos),':',label='PhDOS')
# plt.plot(Freq_final,np.asarray(df.loc[ind].pred_avg)/max(df.loc[ind].pred_avg),'-',label='pred a2f')

# plt.xlabel('Freq (meV)')
# plt.ylabel('Normalized')
# plt.title(df.loc[ind].formula)
# # plt.xlim(0,26)
# plt.legend()

In [ ]:
# ind = 1643
# plt.plot(df.loc[ind].freq_a2f,np.asarray(df.loc[ind].a2f)/max(df.loc[ind].a2f),'--',label='a2F')

# plt.plot(df.loc[ind].freq,np.asarray(df.loc[ind].phdos)/max(df.loc[ind].phdos),':',label='PhDOS')
# plt.plot(Freq_final,np.asarray(df.loc[ind].pred_avg)/max(df.loc[ind].pred_avg),'-',label='pred a2f')

# plt.xlabel('Freq (meV)')
# plt.ylabel('Normalized')
# plt.title(df.loc[ind].formula)
# # plt.ylim(0,0.12)
# plt.legend()

In [ ]:
# plt.plot(df.loc[ind].freq_a2f,np.asarray(df.loc[ind].a2f),'--',label='a2F')

# plt.plot(Freq_final,np.asarray(df.loc[ind].pred_avg),'-',label='pred a2f')

# plt.xlabel('Freq (meV)')
# plt.ylabel('Normalized')
# plt.title(df.loc[ind].formula)
# # plt.ylim(0,0.12)
# plt.legend()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
# d.set_index('formula')

In [ ]:
d = df[['sgn','formula','eah','tcad','tcad_cpd','tcad_final']]
d = d.set_index('formula')
d.sort_values('tcad_final', ascending = False,inplace=True)

In [ ]:
d.loc[d.sgn==194]

In [ ]:
d.head(20)#.loc[d.eah<=0.1]

In [ ]:
# d = df[['mp_id', 'parent_formula', 'sgn', 'formula','dyn_stable','m3gnet_eah','mp_eah','tcad_m3gnet','tcad_mp','tcad_cpd','tcad_final','natoms']]
# df.sort_values('tcad_final', ascending = False)

In [ ]:
# df['mp_eah'] = df.eah

In [ ]:
df.shape

In [ ]:
df.loc[df.tcad_final > 5][['formula','tcad_final']]#.shape

In [ ]:
df.head(7)

In [ ]:
# ### plt.plot(d.tcad_final.values,d.tcad_cpd.values,'.')
# plt.ylim([5,20])
# plt.xlim([5,20])
# plt.plot([5,20],[5,20])

In [26]:
plt.plot(d.tcad_final.values,d.tcad_cpd.values,'.')
plt.ylim([5,20])
plt.xlim([5,20])
plt.plot([5,20],[5,20])

NameError: name 'd' is not defined

In [43]:
ind = 1655
# plt.plot(df.loc[ind].freq_a2f,np.asarray(df.loc[ind].a2f)/max(df.loc[ind].a2f),'--',label='a2F')
plt.plot(df.loc[ind].freq_a2f,np.asarray(df.loc[ind].a2f),'--',label='a2F')

plt.plot(df.loc[ind].freq,np.asarray(df.loc[ind].phdos)/max(df.loc[ind].phdos),':',label='PhDOS')
# plt.plot(df.loc[ind].freq_a2f,np.asarray(df.loc[ind].pred_avg)/max(df.loc[ind].pred_avg),'-',label='pred a2f')

plt.xlabel('Freq (meV)')
plt.ylabel('a2F')
plt.title(df.loc[ind].formula)
plt.legend()

KeyError: 1655